# DA6401 Assignment 2 — Training Notebook Kaggle

Run each section **in order**.  After training, copy the three `.pth` files to Google Drive
and paste the gdown file IDs into `models/multitask.py`.

## 0. Setup

In [2]:
# Install extra dependencies if not already present
!pip install -q wandb gdown albumentations

In [ ]:
import os, sys

WORK_DIR = "/kaggle/working"          # change if different
DATA_DIR = "/kaggle/input/datasets/govindharshavardhan/dl-data/data"   # change to your dataset path

sys.path.insert(0, WORK_DIR)
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

import wandb
# wandb.login()   # uncomment and run once

Device: cuda


## 1. Shared helpers

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

IMAGE_SIZE  = 224
BATCH_SIZE  = 16
LR          = 1e-4
WEIGHT_DECAY= 1e-4
EPOCHS_CLS  = 75   # classifier
EPOCHS_LOC  = 20   # localizer
EPOCHS_SEG  = 20   # segmenter
DROPOUT_P   = 0.5
NUM_WORKERS = 4
WANDB_PROJECT = "da6401-a2"


def save_ckpt(model, optimizer, epoch, loss, path):
    torch.save({
        "epoch":              epoch,
        "model_state_dict":   model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss":               loss,
    }, path)
    print(f"  Saved {path}")


def accuracy(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()


def dice_score(pred, target, num_classes=3, smooth=1e-7):
    probs    = F.softmax(pred, dim=1)
    pred_cls = probs.argmax(dim=1)
    scores   = []
    for c in range(num_classes):
        p     = (pred_cls == c).float()
        t     = (target == c).float()
        inter = (p * t).sum()
        union = p.sum() + t.sum()
        scores.append(((2 * inter + smooth) / (union + smooth)).item())
    return sum(scores) / num_classes


def mean_iou(pred_boxes, target_boxes, eps=1e-7):
    """pred_boxes, target_boxes: (N,4) in pixel space [cx,cy,w,h]"""
    px1 = pred_boxes[:,0] - pred_boxes[:,2]/2
    py1 = pred_boxes[:,1] - pred_boxes[:,3]/2
    px2 = pred_boxes[:,0] + pred_boxes[:,2]/2
    py2 = pred_boxes[:,1] + pred_boxes[:,3]/2
    tx1 = target_boxes[:,0] - target_boxes[:,2]/2
    ty1 = target_boxes[:,1] - target_boxes[:,3]/2
    tx2 = target_boxes[:,0] + target_boxes[:,2]/2
    ty2 = target_boxes[:,1] + target_boxes[:,3]/2
    inter = (torch.min(px2,tx2)-torch.max(px1,tx1)).clamp(0) * \
            (torch.min(py2,ty2)-torch.max(py1,ty1)).clamp(0)
    union = pred_boxes[:,2]*pred_boxes[:,3] + \
            target_boxes[:,2]*target_boxes[:,3] - inter
    return (inter/(union+eps)).mean().item()

print("Helpers ready.")

Helpers ready.


## 2. Dataloaders

In [2]:
import sys
sys.path.append("/kaggle/input/datasets/govindharshavardhan/code-dl/zip")
import os
os.listdir("/kaggle/input/datasets/govindharshavardhan/code-dl/zip")

['localization_loss.py',
 'multitask.py',
 'layers.py',
 'pets_dataset.py',
 'segmentation.py',
 'localization.py',
 '__init__.py',
 'classification.py',
 'vgg11.py',
 'iou_loss.py']

In [6]:
from pets_dataset import get_dataloaders

# Pre-compute mean/std once and reuse
train_cls, val_cls, MEAN, STD = get_dataloaders(
    DATA_DIR, BATCH_SIZE, "classification", IMAGE_SIZE, NUM_WORKERS
)
print(f"Mean={MEAN}  Std={STD}")

train_loc, val_loc, _, _ = get_dataloaders(
    DATA_DIR, BATCH_SIZE, "localization", IMAGE_SIZE, NUM_WORKERS,
    mean=MEAN, std=STD
)
train_seg, val_seg, _, _ = get_dataloaders(
    DATA_DIR, BATCH_SIZE, "segmentation", IMAGE_SIZE, NUM_WORKERS,
    mean=MEAN, std=STD
)

Computing mean/std: 100%|██████████| 185/185 [00:27<00:00,  6.80it/s]


Computed mean = [0.48122328519821167, 0.449827641248703, 0.39638516306877136]
Computed std  = [0.2646635174751282, 0.2598240375518799, 0.26826879382133484]
[classification] train=5912  val=1478  classes=37
Mean=[0.48122328519821167, 0.449827641248703, 0.39638516306877136]  Std=[0.2646635174751282, 0.2598240375518799, 0.26826879382133484]
[localization] train: dropped 2928 samples with missing annotations (2984 remaining)
[localization] val: dropped 776 samples with missing annotations (702 remaining)
[localization] train=2984  val=702  classes=37
[segmentation] train=5912  val=1478  classes=37


## 3. Task 1 — Train Classifier

In [ ]:
from classification import VGG11Classifier

CKPT_CLS = f"{WORK_DIR}/checkpoints/classifier.pth"

model_cls  = VGG11Classifier(num_classes=37, dropout_p=DROPOUT_P).to(DEVICE)
optim_cls  = torch.optim.Adam(model_cls.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched_cls  = torch.optim.lr_scheduler.StepLR(optim_cls, step_size=7, gamma=0.5)
crit_cls   = nn.CrossEntropyLoss()

wandb.init(project=WANDB_PROJECT, name="classifier", reinit=True)
best_val_acc = 0.0

for epoch in range(1, EPOCHS_CLS + 1):
    #  
    model_cls.train()
    tr_loss = tr_acc = 0.0
    for imgs, labels in tqdm(train_cls, desc=f"[CLS] Ep {epoch}/{EPOCHS_CLS} train"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optim_cls.zero_grad()
        logits = model_cls(imgs)
        loss   = crit_cls(logits, labels)
        loss.backward()
        optim_cls.step()
        tr_loss += loss.item()
        tr_acc  += accuracy(logits, labels)
    tr_loss /= len(train_cls);  tr_acc /= len(train_cls)

    #  Val 
    model_cls.eval()
    va_loss = va_acc = 0.0
    with torch.no_grad():
        for imgs, labels in val_cls:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model_cls(imgs)
            va_loss += crit_cls(logits, labels).item()
            va_acc  += accuracy(logits, labels)
    va_loss /= len(val_cls);  va_acc /= len(val_cls)

    sched_cls.step()
    print(f"Ep {epoch:3d}  tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  "
          f"va_loss={va_loss:.4f}  va_acc={va_acc:.4f}")
    wandb.log({"epoch": epoch, "train/loss": tr_loss, "train/acc": tr_acc,
               "val/loss": va_loss, "val/acc": va_acc})

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        save_ckpt(model_cls, optim_cls, epoch, va_loss, CKPT_CLS)

wandb.finish()
print(f"Best classifier val acc: {best_val_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da25s018 (da25s018-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


...

Ep  74  tr_loss=1.0344  tr_acc=0.6546  va_loss=1.1100  va_acc=0.6532


[CLS] Ep 75/75 train: 100%|██████████| 369/369 [01:16<00:00,  4.83it/s]


Ep  75  tr_loss=1.0219  tr_acc=0.6670  va_loss=1.1061  va_acc=0.6546


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
train/acc,▁▁▂▂▂▃▃▄▄▄▅▅▅▅▆▆▇▇▇▇▇▇██████████████████
train/loss,█▇▇▆▆▅▅▅▄▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▂▂▃▃▄▄▅▆▅▇▇▇▇▇▇▇█▇█████████████████████
val/loss,█▇▇▆▅▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,75
train/acc,0.66701
train/loss,1.02189
val/acc,0.65457
val/loss,1.10609


Best classifier val acc: 0.6579


In [8]:
import wandb

wandb.init(project="da6401-assignment2")
artifact = wandb.Artifact('classifier-model-improved', type='model')
artifact.add_file('/kaggle/working/checkpoints/classifier.pth')
wandb.log_artifact(artifact)
wandb.finish()

In [16]:
# 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

## 4. Task 2 — Train Localizer (backbone from classifier)

In [ ]:
import importlib

import localization
import localization_loss

importlib.reload(localization)
importlib.reload(localization_loss)

from localization import VGG11Localizer
from localization_loss import LocalizationLoss

CKPT_LOC = f"{WORK_DIR}/models/localizer.pth"


model_loc = VGG11Localizer(dropout_p=DROPOUT_P).to(DEVICE)

# Warm-start backbone from the trained classifier
model_loc.load_backbone_weights(CKPT_CLS)

optim_loc = torch.optim.Adam(model_loc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched_loc = torch.optim.lr_scheduler.StepLR(optim_loc, step_size=7, gamma=0.5)
crit_loc  = LocalizationLoss(mse_weight=0.5, iou_weight=0.5)

wandb.init(project=WANDB_PROJECT, name="localizer", reinit=True)
best_val_loss = float("inf")

for epoch in range(1, EPOCHS_LOC + 1):
    #  Train 
    model_loc.train()
    tr_loss = tr_iou = 0.0
    for imgs, bboxes in tqdm(train_loc, desc=f"[LOC] Ep {epoch}/{EPOCHS_LOC} train"):
        imgs, bboxes = imgs.to(DEVICE), bboxes.to(DEVICE)
        optim_loc.zero_grad()
        preds = model_loc(imgs)
        loss  = crit_loc(preds, bboxes, IMAGE_SIZE)
        loss.backward()
        optim_loc.step()
        tr_loss += loss.item()
        tr_iou  += mean_iou(preds.detach(), bboxes)
    tr_loss /= len(train_loc);  tr_iou /= len(train_loc)

    #  Val 
    model_loc.eval()
    va_loss = va_iou = 0.0
    with torch.no_grad():
        for imgs, bboxes in val_loc:
            imgs, bboxes = imgs.to(DEVICE), bboxes.to(DEVICE)
            preds = model_loc(imgs)
            va_loss += crit_loc(preds, bboxes, IMAGE_SIZE).item()
            va_iou  += mean_iou(preds, bboxes)
    va_loss /= len(val_loc);  va_iou /= len(val_loc)

    sched_loc.step()
    print(f"Ep {epoch:3d}  tr_loss={tr_loss:.4f}  tr_iou={tr_iou:.4f}  "
          f"va_loss={va_loss:.4f}  va_iou={va_iou:.4f}")
    wandb.log({"epoch": epoch, "train/loss": tr_loss, "train/iou": tr_iou,
               "val/loss": va_loss, "val/iou": va_iou})

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        save_ckpt(model_loc, optim_loc, epoch, va_loss, CKPT_LOC)

wandb.finish()
print(f"Best localizer val loss: {best_val_loss:.4f}")

  Backbone loaded from /kaggle/working/checkpoints/classifier.pth


[LOC] Ep 1/20 train: 100%|██████████| 186/186 [00:31<00:00,  5.97it/s]


Ep   1  tr_loss=0.2816  tr_iou=0.4553  va_loss=0.2563  va_iou=0.5004
  Saved /kaggle/working/checkpoints/localizer.pth


[LOC] Ep 2/20 train:   5%|▌         | 10/186 [00:02<00:38,  4.63it/s]


KeyboardInterrupt: 

In [ ]:
import wandb

wandb.init(project="da6401-assignment2")
artifact = wandb.Artifact('localizer-model-improved', type='model')
artifact.add_file('/kaggle/working/checkpoints/localizer.pth')
wandb.log_artifact(artifact)
wandb.finish()

## 5. Task 3 — Train Segmenter (VGG11 U-Net)

In [ ]:
from segmentation import VGG11UNet, CombinedSegmentationLoss

CKPT_SEG = f"{WORK_DIR}/checkpoints/unet.pth"

model_seg = VGG11UNet(num_classes=3).to(DEVICE)

# Warm-start backbone from the trained classifier
model_seg.set_encoder_weight(CKPT_CLS)

optim_seg = torch.optim.Adam(model_seg.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched_seg = torch.optim.lr_scheduler.StepLR(optim_seg, step_size=7, gamma=0.5)
crit_seg  = CombinedSegmentationLoss(ce_w=1.0, dice_w=1.0)

wandb.init(project=WANDB_PROJECT, name="unet", reinit=True)
best_dice = 0.0

for epoch in range(1, EPOCHS_SEG + 1):
    #  Train 
    model_seg.train()
    tr_loss = 0.0
    for imgs, masks in tqdm(train_seg, desc=f"[SEG] Ep {epoch}/{EPOCHS_SEG} train"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optim_seg.zero_grad()
        preds = model_seg(imgs)
        loss  = crit_seg(preds, masks)
        loss.backward()
        optim_seg.step()
        tr_loss += loss.item()
    tr_loss /= len(train_seg)

    #  Val 
    model_seg.eval()
    va_loss = va_dice = 0.0
    with torch.no_grad():
        for imgs, masks in val_seg:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model_seg(imgs)
            va_loss += crit_seg(preds, masks).item()
            va_dice += dice_score(preds, masks, num_classes=3)
    va_loss /= len(val_seg);  va_dice /= len(val_seg)

    sched_seg.step()
    print(f"Ep {epoch:3d}  tr_loss={tr_loss:.4f}  "
          f"va_loss={va_loss:.4f}  va_dice={va_dice:.4f}")
    wandb.log({"epoch": epoch, "train/loss": tr_loss,
               "val/loss": va_loss, "val/dice": va_dice})

    if va_dice > best_dice:
        best_dice = va_dice
        save_ckpt(model_seg, optim_seg, epoch, va_loss, CKPT_SEG)

wandb.finish()
print(f"Best UNet val dice: {best_dice:.4f}")

In [ ]:
import wandb

wandb.init(project="da6401-assignment2")
artifact = wandb.Artifact('unet-model-improved', type='model')
artifact.add_file('/kaggle/working/checkpoints/unet.pth')
wandb.log_artifact(artifact)
wandb.finish()

## 6. Upload checkpoints to Google Drive

After training, run the cells below to upload your `.pth` files to Google Drive.
Then open each file in Drive → Share → **Anyone with the link** → copy the file ID
(the long string after `/d/` in the URL) and paste it into `models/multitask.py`.

In [ ]:
# Colab / Kaggle with Drive mounted:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp checkpoints/classifier.pth /content/drive/MyDrive/da6401/
# !cp checkpoints/localizer.pth  /content/drive/MyDrive/da6401/
# !cp checkpoints/unet.pth       /content/drive/MyDrive/da6401/

# Or use gdown in reverse (upload via the Drive API)
# Or just download the files from Kaggle output and upload manually.
print("See comments above for upload instructions.")

## 7. Smoke-test MultiTaskPerceptionModel

In [ ]:
# After pasting gdown IDs into models/multitask.py:
from multitask import MultiTaskPerceptionModel

mt = MultiTaskPerceptionModel(
    classifier_path=CKPT_CLS,
    localizer_path=CKPT_LOC,
    unet_path=CKPT_SEG,
).to(DEVICE).eval()

dummy = torch.randn(2, 3, 224, 224, device=DEVICE)
with torch.no_grad():
    out = mt(dummy)

print("classification :", out["classification"].shape)   # (2, 37)
print("localization   :", out["localization"].shape)     # (2, 4)
print("segmentation   :", out["segmentation"].shape)     # (2, 3, 224, 224)
print("loc range      :", out["localization"].min().item(), out["localization"].max().item())
# loc values should be in [0, 224]
assert out["classification"].shape == (2, 37)
assert out["localization"].shape   == (2, 4)
assert out["segmentation"].shape   == (2, 3, 224, 224)
print("All shape assertions passed.")

# Wandb

# 2.1

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import wandb
import torch
import torch.nn as nn
from tqdm import tqdm



CKPT_CLS = f"/kaggle/input/datasets/govindharshavardhan/classifier/classifier.pth"



# Minimal VGG11 WITHOUT BatchNorm
def _conv_no_bn(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=True),
        nn.ReLU(inplace=True),
    )

class VGG11NoBN(nn.Module):
    def __init__(self, num_classes=37, dropout_p=0.5):
        super().__init__()
        self.block1 = nn.Sequential(_conv_no_bn(3, 64),    nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(_conv_no_bn(64, 128),  nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(_conv_no_bn(128, 256), _conv_no_bn(256, 256), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(_conv_no_bn(256, 512), _conv_no_bn(512, 512), nn.MaxPool2d(2, 2))
        self.block5 = nn.Sequential(_conv_no_bn(512, 512), _conv_no_bn(512, 512), nn.MaxPool2d(2, 2))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.block5(self.block4(self.block3(self.block2(self.block1(x)))))
        return self.classifier(x)

# Activation capture hook
def get_conv3_activations(model_with_bn, model_no_bn, loader, device, n_batches=3):
    acts_bn, acts_no_bn = [], []
    h1 = model_with_bn.block3[0][0].register_forward_hook(lambda m, i, o: acts_bn.append(o.detach().cpu()))
    h2 = model_no_bn.block3[0][0].register_forward_hook(lambda m, i, o: acts_no_bn.append(o.detach().cpu()))
    model_with_bn.eval(); model_no_bn.eval()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches: break
            imgs = imgs.to(device)
            model_with_bn(imgs)
            model_no_bn(imgs)
    h1.remove(); h2.remove()
    return torch.cat(acts_bn).numpy(), torch.cat(acts_no_bn).numpy()

# Load trained BN model
from vgg11 import VGG11Encoder
model_bn = VGG11Encoder(return_features=False).to(DEVICE)
ck   = torch.load(CKPT_CLS, map_location=DEVICE)
sd   = ck.get('model_state_dict', ck)
enc_sd = {k[len('features.'):]: v for k, v in sd.items() if k.startswith('features.')}
if not enc_sd:
    enc_sd = {k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}
model_bn.load_state_dict(enc_sd, strict=False)

# nit no-BN model (random weights — pre-training baseline) 
model_no_bn = VGG11NoBN(num_classes=37, dropout_p=DROPOUT_P).to(DEVICE)

# Section 2.1 — single wandb run
wandb.init(project=WANDB_PROJECT, name='2.1-batchnorm-activations', reinit=True)

# t A: Activation distribution plot (random-init no-BN vs trained BN) 
acts_bn, acts_no_bn_init = get_conv3_activations(model_bn, model_no_bn, val_cls, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, acts, title, color in zip(
    axes,
    [acts_bn.flatten(), acts_no_bn_init.flatten()],
    ['With BatchNorm (trained)', 'Without BatchNorm (random init)'],
    ['steelblue', 'tomato']
):
    clip = np.percentile(np.abs(acts), 99)
    ax.hist(np.clip(acts, -clip, clip), bins=100, color=color, alpha=0.75, density=True)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Activation value')
    ax.set_ylabel('Density')
    ax.axvline(0, color='k', lw=0.8, linestyle='--')
    ax.text(0.97, 0.95, f'mean={acts.mean():.3f}\nstd={acts.std():.3f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

fig.suptitle('3rd Conv Layer Activation Distribution — With vs Without BatchNorm', fontsize=13)
plt.tight_layout()
img_path = "/kaggle/working/activation_dist.png"
plt.savefig(img_path, dpi=120)
wandb.log({"activation_distributions": wandb.Image(img_path)})
plt.close()

# Part B: Train no-BN model, log at epoch 5 and 10
crit      = nn.CrossEntropyLoss()
optim_nbn = torch.optim.Adam(model_no_bn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched_nbn = torch.optim.lr_scheduler.StepLR(optim_nbn, step_size=7, gamma=0.5)

LOG_EPOCHS = {5, 10}
TRAIN_EPOCHS = max(LOG_EPOCHS)   # 10

for epoch in range(1, TRAIN_EPOCHS + 1):
    # Train
    model_no_bn.train()
    tr_loss = tr_acc = 0.0
    for imgs, labels in tqdm(train_cls, desc=f"[NoBN] Ep {epoch}/{TRAIN_EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optim_nbn.zero_grad()
        logits = model_no_bn(imgs)
        loss   = crit(logits, labels)
        loss.backward()
        optim_nbn.step()
        tr_loss += loss.item()
        tr_acc  += accuracy(logits, labels)
    tr_loss /= len(train_cls);  tr_acc /= len(train_cls)

    #  Val 
    model_no_bn.eval()
    va_loss = va_acc = 0.0
    with torch.no_grad():
        for imgs, labels in val_cls:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model_no_bn(imgs)
            va_loss += crit(logits, labels).item()
            va_acc  += accuracy(logits, labels)
    va_loss /= len(val_cls);  va_acc /= len(val_cls)

    sched_nbn.step()
    print(f"[NoBN] Ep {epoch:3d}  tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  "
          f"va_loss={va_loss:.4f}  va_acc={va_acc:.4f}")

    wandb.log({
        "no_bn/epoch":    epoch,
        "no_bn/tr_loss":  tr_loss,
        "no_bn/tr_acc":   tr_acc,
        "no_bn/va_loss":  va_loss,
        "no_bn/va_acc":   va_acc,
    })

    # ivation snapshot at epoch 5 and 10 
    if epoch in LOG_EPOCHS:
        _, acts_no_bn_trained = get_conv3_activations(model_bn, model_no_bn, val_cls, DEVICE)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, acts, title, color in zip(
            axes,
            [acts_bn.flatten(), acts_no_bn_trained.flatten()],
            ['With BatchNorm (trained)', f'Without BatchNorm (epoch {epoch})'],
            ['steelblue', 'tomato']
        ):
            clip = np.percentile(np.abs(acts), 99)
            ax.hist(np.clip(acts, -clip, clip), bins=100, color=color, alpha=0.75, density=True)
            ax.set_title(title, fontsize=12)
            ax.set_xlabel('Activation value')
            ax.set_ylabel('Density')
            ax.axvline(0, color='k', lw=0.8, linestyle='--')
            ax.text(0.97, 0.95, f'mean={acts.mean():.3f}\nstd={acts.std():.3f}',
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

        fig.suptitle(f'Activation Distribution — With BN vs No BN after {epoch} epochs', fontsize=13)
        plt.tight_layout()
        snap_path = f"/kaggle/working/activation_dist_epoch{epoch}.png"
        plt.savefig(snap_path, dpi=120)
        wandb.log({f"activation_dist_epoch{epoch}": wandb.Image(snap_path)})
        plt.close()

wandb.finish()
print("Section 2.1 done (activation analysis + no-BN training).")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da25s018 (da25s018-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


...

[NoBN] Ep   9  tr_loss=3.6113  tr_acc=0.0300  va_loss=3.6111  va_acc=0.0269


[NoBN] Ep 10/10: 100%|██████████| 369/369 [00:45<00:00,  8.15it/s]


[NoBN] Ep  10  tr_loss=3.6112  tr_acc=0.0246  va_loss=3.6111  va_acc=0.0269


no_bn/epoch,▁▂▃▃▄▅▆▆▇█
no_bn/tr_acc,█▄▂▅▂▃▄▃█▁
no_bn/tr_loss,█▅▆▅▃▂▂▄▁▁
no_bn/va_acc,▁▁▁▁▁▁▁▁▁▁
no_bn/va_loss,█▆▄▄▃▃▂▂▁▁
no_bn/epoch,10
no_bn/tr_acc,0.02456
no_bn/tr_loss,3.6112
no_bn/va_acc,0.02688
no_bn/va_loss,3.61106


Section 2.1 done (activation analysis + no-BN training).


# 2.2

In [ ]:
# Train 3 small classifiers (fewer epochs — just enough to show the gap)
from classification import VGG11Classifier

EPOCHS_DROPOUT_STUDY = 15
DROPOUT_CONDITIONS   = [('no_dropout', 0.0), ('dropout_p02', 0.2), ('dropout_p05', 0.5)]

# Store curves for overlay plot
curves = {}   # name -> {'tr': [], 'va': []}

for run_name, dp in DROPOUT_CONDITIONS:
    print(f'\n=== {run_name} (p={dp}) ===')
    m   = VGG11Classifier(num_classes=37, dropout_p=dp).to(DEVICE)
    # warm-start backbone to save time (only heads differ by dropout p)
    m.load_state_dict(torch.load(CKPT_CLS, map_location=DEVICE).get('model_state_dict',
                      torch.load(CKPT_CLS, map_location=DEVICE)), strict=False)
    opt  = torch.optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    tr_losses, va_losses = [], []

    wandb.init(project=WANDB_PROJECT, name=f'2.2-{run_name}', reinit=True)

    for epoch in range(1, EPOCHS_DROPOUT_STUDY + 1):
        m.train()
        tr_l = 0.0
        for imgs, labels in train_cls:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = crit(m(imgs), labels)
            loss.backward(); opt.step()
            tr_l += loss.item()
        tr_l /= len(train_cls)

        m.eval()
        va_l = 0.0
        with torch.no_grad():
            for imgs, labels in val_cls:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                va_l += crit(m(imgs), labels).item()
        va_l /= len(val_cls)

        tr_losses.append(tr_l); va_losses.append(va_l)
        wandb.log({'epoch': epoch, 'train/loss': tr_l, 'val/loss': va_l})
        print(f'  Ep {epoch:2d}  tr={tr_l:.4f}  va={va_l:.4f}')

    wandb.finish()
    curves[run_name] = {'tr': tr_losses, 'va': va_losses}
    del m

# Overlay plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'no_dropout': 'tab:red', 'dropout_p02': 'tab:orange', 'dropout_p05': 'tab:blue'}
labels_map = {'no_dropout': 'No Dropout', 'dropout_p02': 'Dropout p=0.2', 'dropout_p05': 'Dropout p=0.5'}
epochs_x = range(1, EPOCHS_DROPOUT_STUDY + 1)

for k, c in colors.items():
    axes[0].plot(epochs_x, curves[k]['tr'], color=c, label=labels_map[k], lw=2)
    axes[1].plot(epochs_x, curves[k]['va'], color=c, label=labels_map[k], lw=2)

for ax, title in zip(axes, ['Training Loss', 'Validation Loss']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('CrossEntropy Loss')
    ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('Effect of Custom Dropout on Train/Val Loss', fontsize=13)
plt.tight_layout()

wandb.init(project=WANDB_PROJECT, name='2.2-dropout-overlay', reinit=True)
wandb.log({'dropout_loss_curves': wandb.Image(fig)})

# Also log as wandb line chart so curves are interactive
for k in DROPOUT_CONDITIONS:
    name, dp = k
    for ep, (tr, va) in enumerate(zip(curves[name]['tr'], curves[name]['va']), 1):
        wandb.log({f'{name}/train_loss': tr, f'{name}/val_loss': va, 'epoch': ep})

wandb.finish()
plt.savefig('/kaggle/working/dropout_curves.png', dpi=120)
plt.close()
print('Section 2.2 done.')


=== no_dropout (p=0.0) ===


epoch,▁█
train/loss,█▁
val/loss,█▁
epoch,2
train/loss,1.54391
val/loss,1.70931


  Ep  1  tr=1.5571  va=1.7056
  Ep  2  tr=1.4631  va=1.6625
  Ep  3  tr=1.4017  va=1.6311
  Ep  4  tr=1.3101  va=1.5152
  Ep  5  tr=1.2135  va=1.3969
  Ep  6  tr=1.1365  va=1.5884
  Ep  7  tr=1.0921  va=1.2879
  Ep  8  tr=1.0062  va=1.3988
  Ep  9  tr=0.9426  va=1.5173
  Ep 10  tr=0.9031  va=1.4283
  Ep 11  tr=0.8510  va=1.5112
  Ep 12  tr=0.8072  va=1.1627
  Ep 13  tr=0.7544  va=1.2295
  Ep 14  tr=0.6912  va=1.2151
  Ep 15  tr=0.6859  va=1.4851


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/loss,█▇▇▆▅▅▄▄▃▃▂▂▂▁▁
val/loss,█▇▇▆▄▆▃▄▆▄▅▁▂▂▅
epoch,15
train/loss,0.68587
val/loss,1.48505



=== dropout_p02 (p=0.2) ===


...


=== dropout_p05 (p=0.5) ===


  Ep  1  tr=1.8388  va=1.7067
  Ep  2  tr=1.7713  va=2.0309
  Ep  3  tr=1.7440  va=1.6764
  Ep  4  tr=1.7218  va=1.7127
  Ep  5  tr=1.6350  va=1.5161
  Ep  6  tr=1.5928  va=2.1417
  Ep  7  tr=1.5265  va=1.7682
  Ep  8  tr=1.4841  va=1.3617
  Ep  9  tr=1.4532  va=1.3782
  Ep 10  tr=1.3816  va=1.2694
  Ep 11  tr=1.3665  va=1.2394
  Ep 12  tr=1.3211  va=1.2295
  Ep 13  tr=1.2679  va=1.5139
  Ep 14  tr=1.2375  va=1.5489
  Ep 15  tr=1.2290  va=1.3147


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/loss,█▇▇▇▆▅▄▄▄▃▃▂▁▁▁
val/loss,▅▇▄▅▃█▅▂▂▁▁▁▃▃▂
epoch,15
train/loss,1.22905
val/loss,1.31474


dropout_p02/train_loss,█▇▇▆▅▅▄▄▃▃▂▂▂▂▁
dropout_p02/val_loss,█▃█▄▂▃▃▁▂▁▂▃▁▁▂
dropout_p05/train_loss,█▇▇▇▆▅▄▄▄▃▃▂▁▁▁
dropout_p05/val_loss,▅▇▄▅▃█▅▂▂▁▁▁▃▃▂
epoch,▁▁▂▃▃▃▄▅▅▆▇▇▇█▁▁▃▃▃▄▅▅▅▆▇▇█▁▁▂▃▃▄▅▅▅▆▇▇█
no_dropout/train_loss,█▇▇▆▅▅▄▄▃▃▂▂▂▁▁
no_dropout/val_loss,█▇▇▆▄▆▃▄▆▄▅▁▂▂▅
dropout_p02/train_loss,0.87741
dropout_p02/val_loss,1.36631
dropout_p05/train_loss,1.22905
dropout_p05/val_loss,1.31474


Section 2.2 done.


In [13]:
CKPT_LOC = f"/kaggle/input/datasets/govindharshavardhan/models/localizer.pth"
CKPT_SEG = f"/kaggle/input/datasets/govindharshavardhan/models/unet.pth"


# 2.3

In [14]:
import time
from segmentation import VGG11UNet, CombinedSegmentationLoss

EPOCHS_TL = 10   # enough to show convergence differences

def train_seg_strategy(strategy_name, freeze_backbone, unfreeze_last_n_blocks=0):
    """
    strategy_name     : label for logging
    freeze_backbone   : if True freeze all encoder weights first
    unfreeze_last_n_blocks: then unfreeze the last N encoder blocks (0=none, 2=block4+block5)
    """
    model = VGG11UNet(num_classes=3, freeze_backbone=freeze_backbone).to(DEVICE)
    model.set_encoder_weight(CKPT_CLS)

    if freeze_backbone and unfreeze_last_n_blocks > 0:
        # Unfreeze the last N blocks of encoder
        all_blocks = [model.encoder.block1, model.encoder.block2,
                      model.encoder.block3, model.encoder.block4, model.encoder.block5]
        for blk in all_blocks[-unfreeze_last_n_blocks:]:
            for p in blk.parameters(): p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{strategy_name}] trainable params: {trainable:,}')

    opt  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=1e-4, weight_decay=1e-4)
    crit = CombinedSegmentationLoss(ce_w=1.0, dice_w=1.0)

    wandb.init(project=WANDB_PROJECT, name=f'2.3-{strategy_name}', reinit=True)

    best_dice = 0.0
    for epoch in range(1, EPOCHS_TL + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0
        for imgs, masks in tqdm(train_seg, desc=f'[{strategy_name}] Ep {epoch}', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            preds = model(imgs)
            loss  = crit(preds, masks)
            loss.backward(); opt.step()
            tr_loss += loss.item()
        tr_loss /= len(train_seg)
        epoch_time = time.time() - t0

        model.eval()
        va_loss = va_dice = 0.0
        with torch.no_grad():
            for imgs, masks in val_seg:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                va_loss += crit(preds, masks).item()
                va_dice += dice_score(preds, masks, num_classes=3)
        va_loss /= len(val_seg); va_dice /= len(val_seg)
        if va_dice > best_dice: best_dice = va_dice

        wandb.log({'epoch': epoch, 'train/loss': tr_loss,
                   'val/loss': va_loss, 'val/dice': va_dice,
                   'epoch_time_s': epoch_time})
        print(f'  Ep {epoch:2d}  tr={tr_loss:.4f}  va_dice={va_dice:.4f}  t={epoch_time:.1f}s')

    wandb.summary['best_val_dice'] = best_dice
    wandb.finish()
    del model
    return best_dice

print('\n--- Strategy 1: Strict Feature Extractor (all frozen) ---')
d1 = train_seg_strategy('frozen',   freeze_backbone=True,  unfreeze_last_n_blocks=0)

print('\n--- Strategy 2: Partial Fine-Tuning (unfreeze last 2 blocks) ---')
d2 = train_seg_strategy('partial',  freeze_backbone=True,  unfreeze_last_n_blocks=2)

print('\n--- Strategy 3: Full Fine-Tuning ---')
d3 = train_seg_strategy('full',     freeze_backbone=False, unfreeze_last_n_blocks=0)

print(f'\nBest Dice — Frozen: {d1:.4f} | Partial: {d2:.4f} | Full: {d3:.4f}')


--- Strategy 1: Strict Feature Extractor (all frozen) ---
  Encoder loaded from /kaggle/working/checkpoints/classifier.pth
  [frozen] trainable params: 11,195,459


  Ep  1  tr=0.8699  va_dice=0.7767  t=100.6s


  Ep  2  tr=0.6508  va_dice=0.7937  t=101.9s


  Ep  3  tr=0.5962  va_dice=0.8054  t=102.2s


  Ep  4  tr=0.5576  va_dice=0.8095  t=101.1s


...

  Ep  8  tr=0.3973  va_dice=0.8343  t=140.3s


  Ep  9  tr=0.3807  va_dice=0.8443  t=140.3s


  Ep 10  tr=0.3536  va_dice=0.8408  t=140.6s


epoch,▁▂▃▃▄▅▆▆▇█
epoch_time_s,▆▁▄▆█▁▃▅▅▆
train/loss,█▅▄▃▃▂▂▂▁▁
val/dice,▁▃▃▅▆▆▇▇██
val/loss,█▆▆▄▃▃▂▂▁▁
best_val_dice,0.84426
epoch,10
epoch_time_s,140.59495
train/loss,0.35356
val/dice,0.84081
val/loss,0.47599



Best Dice — Frozen: 0.8260 | Partial: 0.8321 | Full: 0.8443


# 2.4

In [ ]:
from classification import VGG11Classifier

# Reload best classifier
model_cls = VGG11Classifier(num_classes=37).to(DEVICE)
model_cls.load_state_dict(torch.load(CKPT_CLS, map_location=DEVICE)
                          .get('model_state_dict',
                               torch.load(CKPT_CLS, map_location=DEVICE)))
model_cls.eval()

# Grab one image
sample_img, sample_label = next(iter(val_cls))
sample_img = sample_img[:1].to(DEVICE)    # (1, 3, 224, 224)

# Register hooks on first and last conv layers
# First conv  = features.block1[0][0]  (Conv2d 3→64)
# Last conv before pool = features.block5[0][3]  (Conv2d 512→512, second in block5)

first_conv_feats = []
last_conv_feats  = []

h1 = model_cls.features.block1[0][0].register_forward_hook(
    lambda m, i, o: first_conv_feats.append(o.detach().cpu()))

h2 = model_cls.features.block5[1][0].register_forward_hook(
    lambda m, i, o: last_conv_feats.append(o.detach().cpu()))

with torch.no_grad():
    _ = model_cls(sample_img)

h1.remove(); h2.remove()

first_maps = first_conv_feats[0][0]   # (64, 224, 224)
last_maps  = last_conv_feats[0][0]    # (512, 7, 7)

# Visualize 16 channels from each layer
def plot_feature_maps(fmaps, title, n=16, cols=8):
    rows = n // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    for i, ax in enumerate(axes.flat):
        if i < n:
            fm = fmaps[i].numpy()
            ax.imshow(fm, cmap='viridis', interpolation='nearest')
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    return fig

fig_first = plot_feature_maps(first_maps, 'First Conv Layer (3→64) — Low-level edges/textures')
fig_last  = plot_feature_maps(last_maps,  'Last Conv Layer (512→512) — High-level semantic features')

# Also show the input image
import torchvision
img_vis = sample_img[0].cpu()
# de-normalise for display
mean_t = torch.tensor(MEAN).view(3,1,1)
std_t  = torch.tensor(STD).view(3,1,1)
img_vis = (img_vis * std_t + mean_t).clamp(0, 1).permute(1,2,0).numpy()

fig_in, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img_vis); ax.set_title('Input image'); ax.axis('off')

wandb.init(project=WANDB_PROJECT, name='2.4-feature-maps', reinit=True)
wandb.log({
    'input_image':       wandb.Image(fig_in),
    'first_conv_fmaps':  wandb.Image(fig_first),
    'last_conv_fmaps':   wandb.Image(fig_last),
})
wandb.finish()
for f in [fig_in, fig_first, fig_last]: plt.close(f)
print('Section 2.4 done.')

Section 2.4 done.


# 2.5

In [19]:
from localization import VGG11Localizer

model_loc = VGG11Localizer().to(DEVICE)
model_loc.load_state_dict(torch.load(CKPT_LOC, map_location=DEVICE)
                          .get('model_state_dict',
                               torch.load(CKPT_LOC, map_location=DEVICE)))
model_loc.eval()

model_cls2 = VGG11Classifier(num_classes=37).to(DEVICE)
model_cls2.load_state_dict(torch.load(CKPT_CLS, map_location=DEVICE)
                           .get('model_state_dict',
                                torch.load(CKPT_CLS, map_location=DEVICE)))
model_cls2.eval()

def compute_single_iou(pred_box, gt_box, eps=1e-7):
    """Both boxes: [cx, cy, w, h] in pixels."""
    px1, py1 = pred_box[0]-pred_box[2]/2, pred_box[1]-pred_box[3]/2
    px2, py2 = pred_box[0]+pred_box[2]/2, pred_box[1]+pred_box[3]/2
    tx1, ty1 = gt_box[0]-gt_box[2]/2,   gt_box[1]-gt_box[3]/2
    tx2, ty2 = gt_box[0]+gt_box[2]/2,   gt_box[1]+gt_box[3]/2
    inter_w  = max(0, min(px2, tx2) - max(px1, tx1))
    inter_h  = max(0, min(py2, ty2) - max(py1, ty1))
    inter    = inter_w * inter_h
    union    = pred_box[2]*pred_box[3] + gt_box[2]*gt_box[3] - inter
    return inter / (union + eps)

def draw_bbox_image(img_np, pred_box, gt_box):
    """img_np: (H,W,3) float [0,1]. Returns wandb.Image with overlaid bboxes."""
    import matplotlib.patches as patches
    fig, ax = plt.subplots(1, figsize=(4, 4))
    ax.imshow(img_np)
    ax.axis('off')
    H, W = img_np.shape[:2]
    for box, color, lbl in [(gt_box, 'lime', 'GT'), (pred_box, 'red', 'Pred')]:
        cx, cy, bw, bh = box
        # pixel coords, clamped
        x0 = max(0, cx - bw/2); y0 = max(0, cy - bh/2)
        bw = min(bw, W - x0);   bh = min(bh, H - y0)
        rect = patches.Rectangle((x0, y0), bw, bh,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0 - 3, lbl, color=color, fontsize=9, fontweight='bold')
    plt.tight_layout()
    img_buf = wandb.Image(fig)
    plt.close(fig)
    return img_buf

mean_t = torch.tensor(MEAN).view(3,1,1)
std_t  = torch.tensor(STD).view(3,1,1)

table = wandb.Table(columns=['image', 'gt_box', 'pred_box', 'confidence', 'iou', 'note'])

collected = 0
TARGET_N  = 12    # collect 12 samples (some will be failure cases)

with torch.no_grad():
    for imgs, bboxes in val_loc:
        if collected >= TARGET_N: break
        imgs_d  = imgs.to(DEVICE)
        preds   = model_loc(imgs_d).cpu()
        logits  = model_cls2(imgs_d).cpu()
        confs   = torch.softmax(logits, dim=1).max(dim=1).values

        for j in range(min(imgs.size(0), TARGET_N - collected)):
            img_vis = (imgs[j] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
            pred_b  = preds[j].numpy()
            gt_b    = bboxes[j].numpy()
            conf    = round(confs[j].item(), 4)
            iou_val = round(compute_single_iou(pred_b, gt_b), 4)
            note    = 'failure: high conf, low IoU' if (conf > 0.7 and iou_val < 0.3) else \
                      'good'   if iou_val > 0.5    else 'poor IoU'
            bbox_img = draw_bbox_image(img_vis, pred_b, gt_b)
            table.add_data(bbox_img,
                           str(np.round(gt_b, 1).tolist()),
                           str(np.round(pred_b, 1).tolist()),
                           conf, iou_val, note)
            collected += 1

wandb.init(project=WANDB_PROJECT, name='2.5-detection-table', reinit=True)
wandb.log({'detection_results': table})
wandb.finish()
print(f'Section 2.5 done. Logged {collected} samples.')

Section 2.5 done. Logged 12 samples.


# 2.6

In [ ]:
from segmentation import VGG11UNet

model_seg = VGG11UNet(num_classes=3).to(DEVICE)
model_seg.load_state_dict(torch.load(CKPT_SEG, map_location=DEVICE)
                          .get('model_state_dict',
                               torch.load(CKPT_SEG, map_location=DEVICE)))
model_seg.eval()

CLASS_NAMES = ['foreground', 'background', 'boundary']
PALETTE     = np.array([[255, 80, 80], [80, 180, 255], [255, 220, 50]], dtype=np.uint8)

def mask_to_rgb(mask_np):
    """mask_np: (H, W) int. Returns (H, W, 3) uint8."""
    out = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c, col in enumerate(PALETTE):
        out[mask_np == c] = col
    return out

# ollect 5 sample images + track metrics over entire val set 
seg_table  = wandb.Table(columns=['original', 'ground_truth', 'prediction',
                                  'pixel_acc', 'dice'])
all_pixel_acc, all_dice = [], []
samples_logged = 0

mean_t = torch.tensor(MEAN).view(3,1,1)
std_t  = torch.tensor(STD).view(3,1,1)

with torch.no_grad():
    for imgs, masks in val_seg:
        imgs_d  = imgs.to(DEVICE)
        masks_d = masks.to(DEVICE)
        preds   = model_seg(imgs_d)
        pred_cls= preds.argmax(dim=1)

        # Pixel accuracy
        pa = (pred_cls == masks_d).float().mean().item()
        dc = dice_score(preds, masks_d, num_classes=3)
        all_pixel_acc.append(pa)
        all_dice.append(dc)

        if samples_logged < 5:
            for j in range(min(imgs.size(0), 5 - samples_logged)):
                img_vis = (imgs[j] * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()
                gt_rgb  = mask_to_rgb(masks[j].numpy())
                pr_rgb  = mask_to_rgb(pred_cls[j].cpu().numpy())

                pa_j = (pred_cls[j].cpu() == masks[j]).float().mean().item()
                dc_j = dice_score(preds[j:j+1], masks_d[j:j+1], num_classes=3)

                def make_seg_img(arr):
                    f, ax = plt.subplots(figsize=(3,3))
                    ax.imshow(arr); ax.axis('off'); plt.tight_layout()
                    wi = wandb.Image(f); plt.close(f); return wi

                seg_table.add_data(
                    make_seg_img(img_vis),
                    make_seg_img(gt_rgb),
                    make_seg_img(pr_rgb),
                    round(pa_j, 4),
                    round(dc_j, 4)
                )
                samples_logged += 1

mean_pa   = np.mean(all_pixel_acc)
mean_dice = np.mean(all_dice)
print(f'Val Pixel Accuracy: {mean_pa:.4f}   Val Dice: {mean_dice:.4f}')

# Scatter plot showing PA vs Dice per batch
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(all_pixel_acc, all_dice, alpha=0.5, s=20, color='steelblue')
ax.axline((0,0), slope=1, color='gray', linestyle='--', lw=1, label='PA = Dice line')
ax.set_xlabel('Pixel Accuracy'); ax.set_ylabel('Dice Score')
ax.set_title('Pixel Accuracy vs Dice Score per val batch')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

wandb.init(project=WANDB_PROJECT, name='2.6-seg-eval', reinit=True)
wandb.log({
    'seg_samples':          seg_table,
    'pa_vs_dice_scatter':   wandb.Image(fig),
    'mean_pixel_accuracy':  mean_pa,
    'mean_dice':            mean_dice,
})
wandb.finish()
plt.close()
print('Section 2.6 done.')

Val Pixel Accuracy: 0.9099   Val Dice: 0.8541


mean_dice,▁
mean_pixel_accuracy,▁
mean_dice,0.85413
mean_pixel_accuracy,0.90986


Section 2.6 done.


# 2.7

In [ ]:
# Download 3 pet images from the web
# Using publicly accessible direct image URLs (Wikimedia / public domain)
import urllib.request
from PIL import Image as PILImage
from torchvision import transforms

NOVEL_URLS = [
    ('dog1',
     'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'),

    ('dog2',
     'https://images.unsplash.com/photo-1518717758536-85ae29035b6d'),

    ('cat1',
     'https://images.unsplash.com/photo-1518791841217-8f162f1e1131')
]

mean_t = torch.tensor(MEAN).view(3,1,1)
std_t  = torch.tensor(STD).view(3,1,1)

preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Reload all three models
from classification import VGG11Classifier
from localization   import VGG11Localizer
from segmentation   import VGG11UNet

m_cls = VGG11Classifier(num_classes=37).to(DEVICE)
m_cls.load_state_dict(torch.load(CKPT_CLS, map_location=DEVICE).get('model_state_dict', {}))
m_cls.eval()

m_loc = VGG11Localizer().to(DEVICE)
m_loc.load_state_dict(torch.load(CKPT_LOC, map_location=DEVICE).get('model_state_dict', {}))
m_loc.eval()

m_seg = VGG11UNet(num_classes=3).to(DEVICE)
m_seg.load_state_dict(torch.load(CKPT_SEG, map_location=DEVICE).get('model_state_dict', {}))
m_seg.eval()

showcase_table = wandb.Table(columns=['name', 'original', 'pipeline_output',
                                       'pred_class', 'confidence', 'bbox_iou_note'])
import matplotlib.patches as patches

for name, url in NOVEL_URLS:
    # Download
    local_path = f'/kaggle/working/{name}.jpg'

    try:
        urllib.request.urlretrieve(url, local_path)
    except Exception as e:
        print(f'Could not download {name}: {e}')
        continue

    # try:
    #     urllib.request.urlretrieve(url, local_path)
    # except Exception as e:
    #     print(f'  Could not download {name}: {e}'); continue

    pil_img = PILImage.open(local_path).convert('RGB')
    tensor  = preprocess(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        cls_logits = m_cls(tensor)
        bbox_pred  = m_loc(tensor)[0].cpu().numpy()
        seg_pred   = m_seg(tensor).argmax(dim=1)[0].cpu().numpy()

    pred_cls   = cls_logits.argmax(1).item()
    confidence = torch.softmax(cls_logits, dim=1)[0, pred_cls].item()

    # Denorm image
    img_vis = (tensor[0].cpu() * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()

    # Build composite figure: original | bbox overlay | seg mask
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_vis); axes[0].set_title('Original'); axes[0].axis('off')

    axes[1].imshow(img_vis)
    cx, cy, bw, bh = bbox_pred
    rect = patches.Rectangle((cx-bw/2, cy-bh/2), bw, bh,
                              linewidth=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].set_title(f'Bbox pred\nconf={confidence:.2f}')
    axes[1].axis('off')

    axes[2].imshow(mask_to_rgb(seg_pred))
    axes[2].set_title('Seg mask\n(R=fg B=bg Y=boundary)')
    axes[2].axis('off')

    fig.suptitle(f'{name}  →  pred class {pred_cls} (conf {confidence:.2f})', fontsize=11)
    plt.tight_layout()

    def fig_to_wandb(f): w = wandb.Image(f); plt.close(f); return w

    # Original image (standalone)
    fig_orig, ax = plt.subplots(figsize=(3,3)); ax.imshow(img_vis); ax.axis('off')
    plt.tight_layout()

    showcase_table.add_data(
        name,
        fig_to_wandb(fig_orig),
        fig_to_wandb(fig),
        pred_cls,
        round(confidence, 4),
        'no GT available — visual inspection only'
    )
    print(f'  {name}: class={pred_cls}  conf={confidence:.3f}')

wandb.init(project=WANDB_PROJECT, name='2.7-pipeline-showcase', reinit=True)
wandb.log({'novel_images_showcase': showcase_table})
wandb.finish()
print('Section 2.7 done.')

  dog1: class=26  conf=0.556
  dog2: class=1  conf=0.414
  cat1: class=5  conf=0.548


Section 2.7 done.


# 2.8

In [ ]:
# Pull run history from wandb API for overlay plots
import wandb

api = wandb.Api()
# runs = api.runs(f"{wandb.api.default_settings['entity']}/{WANDB_PROJECT}")
runs = api.runs(WANDB_PROJECT)

TARGET_RUNS = {
    'classifier': ('train/loss', 'val/loss', 'val/acc'),
    'localizer':  ('train/loss', 'val/loss', 'val/iou'),
    'unet':       ('train/loss', 'val/loss', 'val/dice'),
}

histories = {}
for run in runs:
    if run.name in TARGET_RUNS:
        df = run.history(keys=list(TARGET_RUNS[run.name]) + ['epoch'], pandas=True)
        histories[run.name] = df
        print(f'  Loaded history for run: {run.name} ({len(df)} rows)')

# Plot all training curves in one figure
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
task_configs = [
    ('classifier', 'val/acc',  'Val Accuracy (Classification)'),
    ('localizer',  'val/iou',  'Val IoU (Localization)'),
    ('unet',       'val/dice', 'Val Dice (Segmentation)'),
]

for row, (task, metric, metric_label) in enumerate(task_configs):
    if task not in histories: continue
    df = histories[task]
    ep = df['epoch'] if 'epoch' in df else range(len(df))

    axes[row, 0].plot(ep, df['train/loss'], label='Train loss', color='tab:blue')
    axes[row, 0].plot(ep, df['val/loss'],   label='Val loss',   color='tab:orange')
    axes[row, 0].set_title(f'{task.capitalize()} — Loss'); axes[row, 0].legend(); axes[row, 0].grid(alpha=0.3)

    if metric in df:
        axes[row, 1].plot(ep, df[metric], label=metric_label, color='tab:green')
        axes[row, 1].set_title(metric_label); axes[row, 1].legend(); axes[row, 1].grid(alpha=0.3)

fig.suptitle('Full Training History — All Tasks', fontsize=14)
plt.tight_layout()

wandb.init(project=WANDB_PROJECT, name='2.8-meta-analysis', reinit=True)
wandb.log({'all_tasks_training_curves': wandb.Image(fig)})

# Also re-log each metric properly so wandb panels are interactive
for task, df in histories.items():
    for _, row in df.iterrows():
        log_dict = {'meta_epoch': int(row.get('epoch', 0))}
        for col in df.columns:
            if col != 'epoch' and not col.startswith('_'):
                log_dict[f'{task}/{col}'] = row[col]
        wandb.log(log_dict)

# Summary table
summary_tbl = wandb.Table(columns=['task', 'metric', 'best_val'])

for run in runs:
    if run.name == 'classifier' and 'val/acc' in run.summary:
        summary_tbl.add_data('Classification', 'Accuracy',
                             round(run.summary['val/acc'], 4))

    if run.name == 'localizer' and 'val/iou' in run.summary:
        summary_tbl.add_data('Localization', 'Mean IoU',
                             round(run.summary['val/iou'], 4))

    if run.name == 'unet' and 'val/dice' in run.summary:
        summary_tbl.add_data('Segmentation', 'Dice Score',
                             round(run.summary['val/dice'], 4))
wandb.log({'final_summary': summary_tbl})
wandb.finish()
plt.close()
print('Section 2.8 done. All Part 2 WandB logging complete.')

  Loaded history for run: localizer (20 rows)
  Loaded history for run: unet (20 rows)
  Loaded history for run: classifier (75 rows)


classifier/train/loss,█▇▇▇▆▆▅▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
classifier/val/acc,▁▁▂▃▃▄▄▅▆▅▆▆▆▇▇▇▇▇▇▇████████████████████
classifier/val/loss,██▇▆▆▅▄▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
localizer/train/loss,█▆▅▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁
localizer/val/iou,▁▃▄▅▅▄▆▆▆▆▇▆▇▇████▇█
localizer/val/loss,█▇▅▄▄▅▃▃▂▃▂▃▂▂▁▁▁▁▂▁
meta_epoch,▁▁▁▂▂▂▂▂▃▃▁▂▂▂▂▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▇▇▇▇▇███
unet/train/loss,█▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
unet/val/dice,▁▂▅▅▅▆▄▇▇▇▇▇▇▇██████
unet/val/loss,█▆▃▃▃▂▄▁▁▁▂▁▁▂▁▁▁▂▂▂
classifier/train/loss,1.02189


classifier/train/loss,█▇▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
classifier/val/acc,▁▁▂▃▃▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████████████████████
classifier/val/loss,█▇▇▇▆▅▅▅▄▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
localizer/train/loss,█▆▅▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁
localizer/val/iou,▁▃▄▅▅▄▆▆▆▆▇▆▇▇████▇█
localizer/val/loss,█▇▅▄▄▅▃▃▂▃▂▃▂▂▁▁▁▁▂▁
meta_epoch,▁▁▁▂▂▂▂▂▂▂▃▁▁▁▂▂▂▃▁▁▂▃▃▃▃▃▄▄▄▅▅▅▅▆▆▇▇▇██
unet/train/loss,█▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
unet/val/dice,▁▂▅▅▅▆▄▇▇▇▇▇▇▇██████
unet/val/loss,█▆▃▃▃▂▄▁▁▁▂▁▁▂▁▁▁▂▂▂
classifier/train/loss,1.02189


Section 2.8 done. All Part 2 WandB logging complete.
